# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Task Type: **RANKING** (or CLASSIFICATION / SCORING / CLUSTERING)

### Problem Statement
We are building a ranking system to predict which items (e.g., search results,
recommended content, or job postings) are most relevant to a user query or context.

### Why This Task Type?
- **Input:** A query/context + a list of candidate items
- **Output:** A relevance score or ranked order for each item
- **Goal:** Surface the most relevant items at the top (position matters)
- **Not binary:** Not just "good/bad" but "rank by quality"

### Real-World Action This Supports
When a user searches or requests recommendations, we show them the top-ranked
items first, increasing the likelihood of engagement (clicks, purchases, etc.).

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### What Are We Predicting?

**Primary Target: Relevance Score (0-5 scale)**
- 5 = Highly relevant, user very likely to engage
- 4 = Relevant
- 3 = Somewhat relevant
- 2 = Slightly relevant
- 1 = Not relevant
- 0 = Completely irrelevant

OR

**Alternative Proxy: Click-Through Rate (CTR) or Engagement**
- % of users who clicked on this item when shown
- Time spent on the item
- Number of shares/saves
- Purchase likelihood

### Why This Target?
✓ Observable from user behavior (clicks, dwell time, conversions)
✓ Directly correlates with "goodness"
✓ Can be measured and tracked over time
✓ Actionable for ranking decisions

## 3. Success metric

*One metric you can defend. What number means 'good'?*

### How Do We Measure "Good"?

**Primary Metric: NDCG@10** (Normalized Discounted Cumulative Gain at rank 10)
- What: Measures how well we rank relevant items at the top
- Formula penalizes: Irrelevant items in top positions
- Target: NDCG@10 ≥ 0.75
- Why: Industry standard for ranking; accounts for position importance

**Secondary Metrics:**
- **Mean Reciprocal Rank (MRR):** On average, where does the first relevant item appear?
  - Target: MRR ≥ 0.6
  
- **Precision@5:** Of the top 5 items, how many are relevant?
  - Target: Precision@5 ≥ 0.7
  
- **Recall:** Of all relevant items in the full list, what % do we find in top 10?
  - Target: Recall@10 ≥ 0.65

### Evaluation Protocol
- Split data: 70% train, 15% validation, 15% test
- **Never** evaluate on queries/items in training set
- Use hold-out test set for final metric reporting

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd
import numpy as np

# ============================================================================
# STEP 1: DEFINE THE UNIT OF ANALYSIS
# ============================================================================
# One row = ONE (QUERY, ITEM) PAIR
# This is how ranking models are trained: predict relevance of each item
# for a given query

print("=" * 70)
print("UNIT OF ANALYSIS: ONE ROW = ONE (QUERY, ITEM) PAIR")
print("=" * 70)

# ============================================================================
# STEP 2: CREATE SAMPLE DATA (replace with your actual data)
# ============================================================================

sample_data = {
    'query_id': [1, 1, 1, 1, 1, 2, 2, 2, 2, 2],
    'query_text': [
        'python tutorial', 'python tutorial', 'python tutorial',
        'python tutorial', 'python tutorial',
        'machine learning', 'machine learning', 'machine learning',
        'machine learning', 'machine learning'
    ],
    'item_id': [101, 102, 103, 104, 105, 201, 202, 203, 204, 205],
    'item_title': [
        'Learn Python Basics',
        'Advanced Python',
        'Python for Beginners',
        'Python Tips & Tricks',
        'Python Crash Course',
        'ML Fundamentals',
        'Deep Learning Guide',
        'Statistics for ML',
        'Neural Networks 101',
        'Supervised Learning'
    ],
    'clicks': [45, 12, 38, 5, 22, 120, 85, 30, 95, 50],
    'impressions': [1000, 500, 800, 200, 400, 2000, 1500, 600, 1200, 800],
    'target_relevance': [5, 2, 4, 1, 3, 5, 5, 2, 5, 4]
}

df = pd.DataFrame(sample_data)

# ============================================================================
# STEP 3: FEATURE ENGINEERING
# ============================================================================

# Calculate CTR (Click-Through Rate)
df['ctr'] = df['clicks'] / df['impressions']

# Show the unit of analysis
print("\nDataFrame Head (Unit of Analysis):")
print(df[['query_id', 'query_text', 'item_id', 'item_title',
          'clicks', 'impressions', 'ctr', 'target_relevance']])

print(f"\nShape: {df.shape[0]} rows (query-item pairs), {df.shape[1]} columns")

# ============================================================================
# STEP 4: ANALYZE THE DATA
# ============================================================================

print("\n" + "=" * 70)
print("DATA SUMMARY")
print("=" * 70)
print(f"Total (query, item) pairs: {len(df)}")
print(f"Unique queries: {df['query_id'].nunique()}")
print(f"Unique items: {df['item_id'].nunique()}")
print(f"\nTarget relevance distribution:")
print(df['target_relevance'].value_counts().sort_index())

print(f"\nCTR statistics:")
print(df['ctr'].describe())

# ============================================================================
# STEP 5: VISUALIZE ONE QUERY'S RANKING
# ============================================================================

print("\n" + "=" * 70)
print("EXAMPLE: Ranking for Query 1 ('python tutorial')")
print("=" * 70)

query_1 = df[df['query_id'] == 1][['item_title', 'ctr', 'target_relevance']].sort_values('ctr', ascending=False)
print("\nRanked by CTR (current simple ranking):")
print(query_1.to_string())

print("\n✓ This shows what one query's ranking looks like")
print("✓ Our ML model will learn to predict better rankings than CTR alone")

UNIT OF ANALYSIS: ONE ROW = ONE (QUERY, ITEM) PAIR

DataFrame Head (Unit of Analysis):
   query_id        query_text  item_id            item_title  clicks  \
0         1   python tutorial      101   Learn Python Basics      45   
1         1   python tutorial      102       Advanced Python      12   
2         1   python tutorial      103  Python for Beginners      38   
3         1   python tutorial      104  Python Tips & Tricks       5   
4         1   python tutorial      105   Python Crash Course      22   
5         2  machine learning      201       ML Fundamentals     120   
6         2  machine learning      202   Deep Learning Guide      85   
7         2  machine learning      203     Statistics for ML      30   
8         2  machine learning      204   Neural Networks 101      95   
9         2  machine learning      205   Supervised Learning      50   

   impressions       ctr  target_relevance  
0         1000  0.045000                 5  
1          500  0.024000      

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML Beats a Fixed Rule Here

### Could We Just Use a Simple Rule?

**The Problem:** Ranking relevance is NOT a simple rule. Here's why:

#### 1. **Query-Item Interactions Are Complex**
```python
# NAIVE RULE (fails): "Rank by total clicks"
if item.clicks > 100:
    rank = 1  # Top
else:
    rank = 2  # Bottom
```

**Why this fails:**
- Item A has 120 clicks for query "machine learning"
- Item A has 0 clicks for query "python tutorial"
- Same item, different relevance!
- A fixed rule ignores the query entirely

#### 2. **Multiple Signals Matter, But Their Weights Change**
Relevance depends on:
- **Text similarity** (does item match query keywords?)
- **Engagement history** (clicks, time spent, CTR)
- **Item quality** (rating, freshness, authority)
- **User context** (user's past history, device, location)
- **Popularity** (trending? new?)

It's impossible to hand-tune weights for all combinations:
```python
# BROKEN APPROACH: Hardcoded weights
score = (0.3 * clicks) + (0.2 * ctr) + (0.5 * similarity)
# ^ These weights don't generalize across queries
```

#### 3. **Context Changes Over Time**
- Seasonal trends (Python spikes in fall when courses start)
- New items appear (no historical data yet)
- User preferences shift (relevance changes)
- A fixed rule can't adapt

#### 4. **The "New Item" Problem**
- New Python tutorials have 0 clicks
- A rule like "rank by clicks" will bury them
- ML can use features (text, metadata) to predict relevance without clicks

### **Why ML Solves This**

ML learns a **non-linear ranking function** from labeled data:

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 6. Self-Check ✓

### Have I Covered Everything?

| Requirement | Status | Details |
|---|---|---|
| **ML Task Type** | ✅ | RANKING — predict relevance scores for (query, item) pairs |
| **Target/Proxy** | ✅ | Relevance score (0-5) or CTR; observable from user behavior |
| **Success Metric** | ✅ | NDCG@10 ≥ 0.75; also track Precision@5, MRR |
| **Unit of Analysis** | ✅ | One row = one (query, item) pair; shown in DataFrame above |
| **Why ML > Rule** | ✅ | Complex interactions, multiple signals, adapts over time |
| **Action Connected** | ✅ | Top-ranked items → shown to users → increased engagement |
| **Data Loaded** | ✅ | Sample data shown; replace with actual data from `flyrank/` |

### How This Connects to the ML Loop